In [1]:
import cv2
from ultralytics import YOLO
import os
import time
import datetime

MODEL_PATH = 'model8.pt'
INPUT_VIDEO_PATHS = ['input videos/alles.mp4', 'input videos/front_fixed.mp4'] 
OUTPUT_DIR = 'output videos/' + MODEL_PATH
CONFIDENCE_THRESHOLD = 0.5
OUTPUT_FRAMERATE = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)

for INPUT_VIDEO_PATH in INPUT_VIDEO_PATHS:
    print(f"\nProcessing video: {INPUT_VIDEO_PATH}")
    if not os.path.exists(INPUT_VIDEO_PATH):
        print(f"Error: Input video file not found at {INPUT_VIDEO_PATH}. Skipping.")
        continue

    print(f"Loading model from {MODEL_PATH}...")
    cap = None
    out = None
    
    model = YOLO(MODEL_PATH)
    print("Model loaded successfully!")

    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
    if not cap.isOpened():
        print(f"Error: Could not open video file: {INPUT_VIDEO_PATH}. Skipping.")
        continue

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    input_fps = cap.get(cv2.CAP_PROP_FPS)
    
    print(f"Input video properties: {frame_width}x{frame_height} @ {input_fps:.2f} FPS")

    frame_select_interval = -(-input_fps // OUTPUT_FRAMERATE)
    if frame_select_interval < 1:
        frame_select_interval = 1
    
    print(f"Target output FPS: {OUTPUT_FRAMERATE}. Selecting 1 out of every {frame_select_interval} input frames.")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    base_input_name = os.path.splitext(os.path.basename(INPUT_VIDEO_PATH))[0]
    output_video_filename = os.path.join(OUTPUT_DIR, f'{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}_{base_input_name}_annotated_fps{int(OUTPUT_FRAMERATE)}.mp4')
    
    out = cv2.VideoWriter(output_video_filename, fourcc, OUTPUT_FRAMERATE, (frame_width, frame_height))

    print("Starting video processing...")
    start_time = time.time()
    
    input_frame_idx = 0   
    output_frame_count = 0

    for results in model.predict(source=INPUT_VIDEO_PATH, conf=CONFIDENCE_THRESHOLD, stream=True, verbose=False):
        if input_frame_idx % frame_select_interval == 0:
            annotated_frame = results.plot() 
            out.write(annotated_frame)       
            output_frame_count += 1

        input_frame_idx += 1
    
    end_time = time.time()
    total_time = end_time - start_time
    avg_fps_processing = input_frame_idx / total_time if total_time > 0 else 0

    print(f"\nFinished processing {INPUT_VIDEO_PATH}.")
    print(f"Total input frames processed by YOLO: {input_frame_idx}")
    print(f"Total output frames written: {output_frame_count}")
    print(f"Total processing time: {total_time:.2f} seconds (Avg YOLO processing FPS: {avg_fps_processing:.2f})")
        
    print(f"Annotated video saved successfully to {output_video_filename}")

    if cap is not None and cap.isOpened():
        cap.release()
    if out is not None and out.isOpened():
        out.release()

print("\nAll videos processed.")

c:\Users\larsl\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



Processing video: input videos/alles.mp4
Loading model from model8.pt...
Model loaded successfully!
Input video properties: 848x480 @ 30.00 FPS
Target output FPS: 2. Selecting 1 out of every 15.0 input frames.
Starting video processing...

Finished processing input videos/alles.mp4.
Total input frames processed by YOLO: 2986
Total output frames written: 200
Total processing time: 173.75 seconds (Avg YOLO processing FPS: 17.19)
Annotated video saved successfully to output videos/model8.pt\20250524_135339_alles_annotated_fps2.mp4

Processing video: input videos/front_fixed.mp4
Loading model from model8.pt...
Model loaded successfully!
Input video properties: 848x480 @ 30.00 FPS
Target output FPS: 2. Selecting 1 out of every 15.0 input frames.
Starting video processing...

Finished processing input videos/front_fixed.mp4.
Total input frames processed by YOLO: 3971
Total output frames written: 265
Total processing time: 233.07 seconds (Avg YOLO processing FPS: 17.04)
Annotated video saved